# Phase 4: features + Ridge

Models that *learn* the rule from train (2016-2023), scored on 2024 val. Bar to beat: carry-over, weighted val MAE 0.231. Run `.venv/bin/python -m src.data.load` once first if `data/raw/` is empty.

In [1]:
import sys
from pathlib import Path

# Let the notebook import code from src/ (the notebook lives one folder down).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
from src.data.load import load_pbp
from src.features.qb_games import qb_games
from src.features.target import build_targets
from src.features.history import BASE_FEATURES, base_features
from src.models.baselines import carry_over, league_average
from src.models.linear import fit_linear, predict_linear
from src.models.evaluate import bootstrap_compare, mae

pbp = load_pbp(list(range(2016, 2026)))
games = qb_games(pbp)           # history: every QB-game, any size
targets = build_targets(games)  # what we predict and score
league_avg = league_average(targets.filter(pl.col("split") == "train"))

## Concept 1: linear regression learns the blend

Carry-over mixes this season and last season with a hand-picked K=300. Linear regression gets the same ingredients and learns one weight per feature from train: `prediction = w0 + w1*std_avg + w2*std_dropbacks + w3*last_avg + w4*last_dropbacks`.

In [2]:
feats = base_features(games, targets, league_avg)
train = feats.filter(pl.col("split") == "train")
val = feats.filter(pl.col("split") == "val")

model = fit_linear(train, BASE_FEATURES)
print(f"intercept {model.intercept_:+.4f}")
for name, w in zip(BASE_FEATURES, model.coef_):
    print(f"{name:15s} {w:+.6f}")

val_pred = predict_linear(model, val, BASE_FEATURES)
carry = carry_over(games, val, league_avg, k=300).select("passer_id", "season", "week", pl.col("prediction").alias("carry_over"))
val_all = val_pred.join(carry, on=["passer_id", "season", "week"])
print("\nlinear     val MAE", mae(val_pred))
print("carry_over val MAE", mae(val_all, "carry_over"))
print("bootstrap  ", bootstrap_compare(val_all, "carry_over", "prediction"))

intercept -0.0215
std_avg         +0.202406
std_dropbacks   +0.000071
last_avg        +0.165275
last_dropbacks  +0.000098

linear     val MAE {'weighted': 0.23089889670731564, 'unweighted': 0.25996496360583543}
carry_over val MAE {'weighted': 0.23116817069644627, 'unweighted': 0.2608993333893051}
bootstrap   {'gain': 0.0002692739891306036, 'win_rate': 0.5452, 'low': -0.005723974487672697, 'high': 0.006171402224602971}


### ▶ Play: Concept 1
Choose which ingredients the model gets and see how the learned weights and val MAE change.

In [3]:
# ---- KNOBS -----------------------------------------------------------------
FEATURES = ["std_avg", "last_avg"]  # Try ["std_avg"] alone, or add "std_dropbacks" / "last_dropbacks" back.
WEEK = 8                            # Which 2024 week to show example QBs from.
# ---------------------------------------------------------------------------

m = fit_linear(train, FEATURES)
p = predict_linear(m, val, FEATURES)
print("weights:", {n: round(float(w), 4) for n, w in zip(FEATURES, m.coef_)}, "intercept", round(float(m.intercept_), 4))
print(f"val MAE  default (all 4): {mae(val_pred)['weighted']:.4f}   now: {mae(p)['weighted']:.4f}")
show = ["L.Jackson", "J.Allen", "B.Nix", "H.Hooker"]
print(p.filter(pl.col("week") == WEEK, pl.col("passer").is_in(show))
       .join(val_pred.select("passer_id", "week", pl.col("prediction").alias("default_pred")), on=["passer_id", "week"])
       .select("passer", "std_avg", "std_dropbacks", "last_avg", "default_pred", "prediction", "target"))

weights: {'std_avg': 0.2282, 'last_avg': 0.2212} intercept 0.0262
val MAE  default (all 4): 0.2309   now: 0.2351
shape: (4, 7)
┌───────────┬───────────┬───────────────┬──────────┬──────────────┬────────────┬───────────┐
│ passer    ┆ std_avg   ┆ std_dropbacks ┆ last_avg ┆ default_pred ┆ prediction ┆ target    │
│ ---       ┆ ---       ┆ ---           ┆ ---      ┆ ---          ┆ ---        ┆ ---       │
│ str       ┆ f64       ┆ u32           ┆ f64      ┆ f64          ┆ f64        ┆ f64       │
╞═══════════╪═══════════╪═══════════════╪══════════╪══════════════╪════════════╪═══════════╡
│ L.Jackson ┆ 0.356163  ┆ 229           ┆ 0.121667 ┆ 0.149178     ┆ 0.134407   ┆ 0.250482  │
│ J.Allen   ┆ 0.309913  ┆ 218           ┆ 0.199235 ┆ 0.161239     ┆ 0.141011   ┆ 0.325827  │
│ B.Nix     ┆ -0.104731 ┆ 259           ┆ 0.05556  ┆ -0.015079    ┆ 0.014618   ┆ 0.335967  │
│ H.Hooker  ┆ -1.117706 ┆ 3             ┆ 0.05556  ┆ -0.238364    ┆ -0.21652   ┆ -0.782364 │
└───────────┴───────────┴───────────

## Your exploration

Add cells below.